# ETL do Dataset Olist

Este notebook executa a etapa de ETL dos CSVs da pasta `dataset`.

Nesta fase, o objetivo e limpar, padronizar, transformar e integrar os dados brutos. A modelagem de fatos e dimensoes fica para uma etapa futura.

In [1]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'sqlalchemy': 'sqlalchemy',
    'psycopg2': 'psycopg2-binary',
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print('Instalando dependencias ausentes:', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
else:
    print('Dependencias ja instaladas.')


Instalando dependencias ausentes: psycopg2-binary
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 9.4 MB/s eta 0:00:00


## 1. Importacao de bibliotecas e configuracoes

In [2]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [3]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATASET_DIR = BASE_DIR / 'dataset'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

DATASET_DIR, OUTPUT_DIR

(PosixPath('/home/jovyan/work/dataset'), PosixPath('/home/jovyan/work/output'))

## 2. Funcoes auxiliares

In [4]:
UFS_BRASIL = {
    'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG',
    'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO'
}

def remover_acentos(valor):
    if pd.isna(valor):
        return valor
    texto = str(valor)
    texto = unicodedata.normalize('NFKD', texto)
    return ''.join(char for char in texto if not unicodedata.combining(char))

def padronizar_texto(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.lower()
        .map(remover_acentos)
        .astype('string')
    )

def padronizar_uf(serie):
    uf = serie.astype('string').str.strip().str.upper()
    return uf.where(uf.isin(UFS_BRASIL), pd.NA)

def padronizar_cep(serie):
    return serie.astype('string').str.extract(r'(\d+)', expand=False).str.zfill(5)

def converter_datas(df, colunas):
    for coluna in colunas:
        df[coluna] = pd.to_datetime(df[coluna], errors='coerce')
    return df

def remover_duplicados(df, nome):
    antes = len(df)
    df = df.drop_duplicates().copy()
    depois = len(df)
    print(f'{nome}: {antes - depois} duplicados removidos')
    return df

def resumo_tabela(nome, df):
    return {
        'tabela': nome,
        'linhas': len(df),
        'colunas': df.shape[1],
        'duplicados': int(df.duplicated().sum()),
        'nulos_total': int(df.isna().sum().sum())
    }

## 3. Extracao dos CSVs brutos

In [5]:
arquivos = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

dfs = {
    nome: pd.read_csv(DATASET_DIR / arquivo)
    for nome, arquivo in arquivos.items()
}

for nome, df in dfs.items():
    print(f'{nome}: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

customers: 99,441 linhas x 5 colunas
geolocation: 1,000,163 linhas x 5 colunas
orders: 99,441 linhas x 8 colunas
order_items: 112,650 linhas x 7 colunas
payments: 103,886 linhas x 5 colunas
reviews: 99,224 linhas x 7 colunas
products: 32,951 linhas x 9 colunas
sellers: 3,095 linhas x 4 colunas
category_translation: 71 linhas x 2 colunas


## 4. Perfilamento inicial

In [6]:
perfil_inicial = pd.DataFrame([
    resumo_tabela(nome, df) for nome, df in dfs.items()
])
perfil_inicial

,tabela,linhas,colunas,duplicados,nulos_total
0,customers,99441,5,0,0
1,geolocation,1000163,5,261831,0
2,orders,99441,8,0,4908
3,order_items,112650,7,0,0
4,payments,103886,5,0,0
5,reviews,99224,7,0,145903
6,products,32951,9,0,2448
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


In [7]:
for nome, df in dfs.items():
    print('\n' + '=' * 80)
    print(nome)
    display(df.head(3))
    display(df.isna().sum().sort_values(ascending=False).head(10))


customers


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


geolocation


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64


category_translation


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


product_category_name            0
product_category_name_english    0
dtype: int64

## 5. Transformacao e limpeza das tabelas

In [8]:
customers = remover_duplicados(dfs['customers'], 'customers')
customers['customer_zip_code_prefix'] = padronizar_cep(customers['customer_zip_code_prefix'])
customers['customer_city'] = padronizar_texto(customers['customer_city'])
customers['customer_state'] = padronizar_uf(customers['customer_state'])

sellers = remover_duplicados(dfs['sellers'], 'sellers')
sellers['seller_zip_code_prefix'] = padronizar_cep(sellers['seller_zip_code_prefix'])
sellers['seller_city'] = padronizar_texto(sellers['seller_city'])
sellers['seller_state'] = padronizar_uf(sellers['seller_state'])

customers: 0 duplicados removidos
sellers: 0 duplicados removidos


In [9]:
orders = remover_duplicados(dfs['orders'], 'orders')
orders['order_status'] = padronizar_texto(orders['order_status'])
orders = converter_datas(orders, [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

orders['ano_pedido'] = orders['order_purchase_timestamp'].dt.year
orders['mes_pedido'] = orders['order_purchase_timestamp'].dt.month
orders['trimestre_pedido'] = orders['order_purchase_timestamp'].dt.quarter
orders['dia_semana_pedido'] = orders['order_purchase_timestamp'].dt.day_name()

orders['dias_entrega'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days
orders['dias_estimados_entrega'] = (
    orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']
).dt.days
orders['atraso_dias'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days
orders['atraso_dias'] = orders['atraso_dias'].clip(lower=0)
orders['entregue_no_prazo'] = np.where(orders['atraso_dias'].fillna(0) == 0, 1, 0)

orders: 0 duplicados removidos


In [10]:
order_items = remover_duplicados(dfs['order_items'], 'order_items')
order_items = converter_datas(order_items, ['shipping_limit_date'])

for coluna in ['price', 'freight_value']:
    order_items[coluna] = pd.to_numeric(order_items[coluna], errors='coerce')
    order_items.loc[order_items[coluna] < 0, coluna] = np.nan
    order_items[coluna] = order_items[coluna].fillna(0)

order_items['valor_total_item'] = order_items['price'] + order_items['freight_value']

order_items: 0 duplicados removidos


In [11]:
products = remover_duplicados(dfs['products'], 'products')
translation = remover_duplicados(dfs['category_translation'], 'category_translation')

products['product_category_name'] = padronizar_texto(products['product_category_name']).fillna('sem_categoria')
translation['product_category_name'] = padronizar_texto(translation['product_category_name'])
translation['product_category_name_english'] = padronizar_texto(translation['product_category_name_english'])

products = products.merge(translation, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna(products['product_category_name'])

colunas_numericas_produto = [
    'product_name_lenght', 'product_description_lenght', 'product_photos_qty',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'
]

for coluna in colunas_numericas_produto:
    products[coluna] = pd.to_numeric(products[coluna], errors='coerce')
    products.loc[products[coluna] < 0, coluna] = np.nan
    products[coluna] = products[coluna].fillna(products[coluna].median())
    
products['product_volume_cm3'] = (
    products['product_length_cm'] * products['product_height_cm'] * products['product_width_cm']
)

products: 0 duplicados removidos
category_translation: 0 duplicados removidos


In [12]:
payments = remover_duplicados(dfs['payments'], 'payments')
payments['payment_type'] = padronizar_texto(payments['payment_type'])

payments['payment_installments'] = pd.to_numeric(payments['payment_installments'], errors='coerce').fillna(0)
payments['payment_value'] = pd.to_numeric(payments['payment_value'], errors='coerce').fillna(0)
payments.loc[payments['payment_installments'] < 0, 'payment_installments'] = 0
payments.loc[payments['payment_value'] < 0, 'payment_value'] = 0

payments_tratado = payments.copy()
# A limpeza de nulos e valores negativos já feita acima é suficiente.
# Não faremos o groupby.

payments: 0 duplicados removidos


In [13]:
reviews = remover_duplicados(dfs['reviews'], 'reviews')
reviews = converter_datas(reviews, ['review_creation_date', 'review_answer_timestamp'])
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('').astype('string').str.strip()
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('').astype('string').str.strip()
reviews['review_score'] = pd.to_numeric(reviews['review_score'], errors='coerce')
reviews.loc[~reviews['review_score'].between(1, 5), 'review_score'] = np.nan

reviews['tem_comentario'] = np.where(reviews['review_comment_message'].str.len() > 0, 1, 0)

reviews_tratado = reviews.copy()
# Mantém a granularidade de review_id

reviews: 0 duplicados removidos


In [14]:
geolocation = remover_duplicados(dfs['geolocation'], 'geolocation')
geolocation['geolocation_zip_code_prefix'] = padronizar_cep(geolocation['geolocation_zip_code_prefix'])
geolocation['geolocation_city'] = padronizar_texto(geolocation['geolocation_city'])
geolocation['geolocation_state'] = padronizar_uf(geolocation['geolocation_state'])

geolocation['geolocation_lat'] = pd.to_numeric(geolocation['geolocation_lat'], errors='coerce')
geolocation['geolocation_lng'] = pd.to_numeric(geolocation['geolocation_lng'], errors='coerce')

geolocation_agregado = (
    geolocation.groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean'),
        geolocation_city=('geolocation_city', lambda s: s.mode().iat[0] if not s.mode().empty else pd.NA),
        geolocation_state=('geolocation_state', lambda s: s.mode().iat[0] if not s.mode().empty else pd.NA)
    )
)

geolocation: 261831 duplicados removidos


## 6. Integracao dos dados tratados

In [15]:
## 6. Integracao dos dados tratados
geo_cliente = geolocation_agregado.add_prefix('customer_')
geo_vendedor = geolocation_agregado.add_prefix('seller_')

pedidos_itens = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(order_items, on='order_id', how='left')
    .merge(products, on='product_id', how='left')
    .merge(sellers, on='seller_id', how='left')
    .merge(payments_tratado, on='order_id', how='left') # Atualizado
    .merge(reviews_tratado, on='order_id', how='left')   # Atualizado
    .merge(geo_cliente, left_on='customer_zip_code_prefix', right_on='customer_geolocation_zip_code_prefix', how='left')
    .merge(geo_vendedor, left_on='seller_zip_code_prefix', right_on='seller_geolocation_zip_code_prefix', how='left')
)

# Removidos os fillna() que tentavam acessar as colunas agregadas que não existem mais.
# As validações finais e exportações funcionarão normalmente agora.

print(f"Dimensões finais do OBT (Tabelão): {pedidos_itens.shape}")

Dimensões finais do OBT (Tabelão): (119143, 61)


## 7. Validacoes finais

In [16]:
tabelas_tratadas = {
    'customers_tratado': customers,
    'orders_tratado': orders,
    'order_items_tratado': order_items,
    'products_tratado': products,
    'sellers_tratado': sellers,
    'payments_tratado': payments_tratado, # Atualizado
    'reviews_tratado': reviews_tratado,   # Atualizado
    'geolocation_agregado': geolocation_agregado,
    'pedidos_itens_tratado': pedidos_itens # Pode ser mantido apenas para validações simples
}
relatorio_qualidade = pd.DataFrame([
    resumo_tabela(nome, df) for nome, df in tabelas_tratadas.items()
])
relatorio_qualidade

,tabela,linhas,colunas,duplicados,nulos_total
0,customers_tratado,99441,5,0,0
1,orders_tratado,99441,16,0,10838
2,order_items_tratado,112650,8,0,0
3,products_tratado,32951,11,0,0
4,sellers_tratado,3095,4,0,0
5,payments_tratado,103886,5,0,0
6,reviews_tratado,99224,8,0,0
7,geolocation_agregado,19015,5,0,0
8,pedidos_itens_tratado,119143,61,0,43277


In [17]:
validacoes = {
    'pedidos_sem_cliente': int(pedidos_itens['customer_id'].isna().sum()),
    'itens_sem_produto': int(pedidos_itens['product_id'].isna().sum()),
    'precos_negativos': int((pedidos_itens['price'].fillna(0) < 0).sum()),
    'fretes_negativos': int((pedidos_itens['freight_value'].fillna(0) < 0).sum()),
    'ufs_clientes_invalidas': int(customers['customer_state'].isna().sum()),
    'ufs_vendedores_invalidas': int(sellers['seller_state'].isna().sum())
}

pd.DataFrame(list(validacoes.items()), columns=['validacao', 'valor'])

,validacao,valor
0,pedidos_sem_cliente,0
1,itens_sem_produto,833
2,precos_negativos,0
3,fretes_negativos,0
4,ufs_clientes_invalidas,0
5,ufs_vendedores_invalidas,0


## 8. Gravacao dos arquivos tratados

In [18]:
for nome, df in tabelas_tratadas.items():
    caminho = OUTPUT_DIR / f'{nome}.csv'
    df.to_csv(caminho, index=False)
    print(f'Arquivo gerado: {caminho}')

relatorio_qualidade.to_csv(OUTPUT_DIR / 'relatorio_qualidade_etl.csv', index=False)
print('ETL concluido com sucesso.')

Arquivo gerado: /home/jovyan/work/output/customers_tratado.csv
Arquivo gerado: /home/jovyan/work/output/orders_tratado.csv
Arquivo gerado: /home/jovyan/work/output/order_items_tratado.csv
Arquivo gerado: /home/jovyan/work/output/products_tratado.csv
Arquivo gerado: /home/jovyan/work/output/sellers_tratado.csv
Arquivo gerado: /home/jovyan/work/output/payments_tratado.csv
Arquivo gerado: /home/jovyan/work/output/reviews_tratado.csv
Arquivo gerado: /home/jovyan/work/output/geolocation_agregado.csv
Arquivo gerado: /home/jovyan/work/output/pedidos_itens_tratado.csv
ETL concluido com sucesso.


## 9. Analises exploratorias simples apos o ETL

Estas analises nao substituem a modelagem dimensional. Elas servem apenas para validar que a base tratada ja permite perguntas gerenciais.

In [19]:
faturamento_mes = (
    pedidos_itens.groupby(['ano_pedido', 'mes_pedido'], as_index=False)
    .agg(faturamento=('valor_total_item', 'sum'), pedidos=('order_id', 'nunique'))
    .sort_values(['ano_pedido', 'mes_pedido'])
)
faturamento_mes.head(12)

,ano_pedido,mes_pedido,faturamento,pedidos
0,2016,9,354.75,4
1,2016,10,"58,730.85",324
2,2016,12,19.62,1
3,2017,1,"148,030.11",800
4,2017,2,"303,648.31",1780
5,2017,3,"459,778.64",2682
6,2017,4,"449,707.81",2404
7,2017,5,"634,762.22",3700
8,2017,6,"531,052.06",3245
9,2017,7,"631,341.57",4026


In [20]:
categorias_mais_vendidas = (
    pedidos_itens.groupby('product_category_name_english', as_index=False)
    .agg(quantidade_itens=('order_item_id', 'count'), receita=('valor_total_item', 'sum'))
    .sort_values('receita', ascending=False)
    .head(10)
)
categorias_mais_vendidas

,product_category_name_english,quantidade_itens,receita
43,health_beauty,10032,"1,491,397.76"
73,watches_gifts,6213,"1,358,845.59"
7,bed_bath_table,11988,"1,327,662.02"
68,sports_leisure,9004,"1,205,197.85"
15,computers_accessories,8150,"1,104,362.03"
39,furniture_decor,8832,"955,367.22"
49,housewares,7380,"823,623.50"
20,cool_stuff,3999,"752,702.21"
5,auto,4400,"714,431.95"
42,garden_tools,4590,"625,387.31"


In [21]:
entrega_por_estado = (
    pedidos_itens.groupby('customer_state', as_index=False)
    .agg(
        pedidos=('order_id', 'nunique'),
        dias_entrega_medio=('dias_entrega', 'mean'),
        atraso_medio=('atraso_dias', 'mean'),
        avaliacao_media=('review_score', 'mean') # <-- NOME DA COLUNA CORRIGIDO AQUI
    )
    .sort_values('pedidos', ascending=False)
)
entrega_por_estado.head(10)

,customer_state,pedidos,dias_entrega_medio,atraso_medio,avaliacao_media
25,SP,41746,8.27,0.36,4.11
18,RJ,12852,14.77,1.54,3.80
10,MG,11635,11.50,0.36,4.07
22,RS,5466,14.70,0.62,4.03
17,PR,5045,11.52,0.32,4.09
23,SC,3637,14.51,0.72,3.99
4,BA,3380,18.73,1.40,3.81
6,DF,2140,12.50,0.46,3.99
7,ES,2033,15.24,1.16,3.97
8,GO,2020,14.90,0.73,3.98
